# Reading the KAAUH adult EEG dataset

This notebook shows how to read the three parts of the dataset: the continuous recordings (MNE-Python FIF files), the annotation workbook, and the 1 s windows (`eeg_dataset.npz`).

Requirements: Python 3.9+, `mne`, `numpy`, `pandas`, `openpyxl`, `matplotlib`.

In [ ]:
# !pip install mne numpy pandas openpyxl matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

ROOT = Path("path/to/dataset")          # <-- set to the dataset root folder
FIF_DIR = ROOT / "EEG_data"
XLSX = ROOT / "annotations" / "EEG_annotations.xlsx"
NPZ = ROOT / "windows" / "eeg_dataset.npz"
INDEX = ROOT / "windows" / "recording_index.tsv"

CLASS_NAMES = {"A": "normal", "B": "slowing", "C": "epileptiform discharges", "D": "seizure"}

## 1. Annotation workbook

Sheets: `README`, `Annotations` (one row per event), `Recordings` (one row per recording) and `Codes` (the 26 annotation codes).

In [ ]:
ann = pd.read_excel(XLSX, sheet_name="Annotations")
rec = pd.read_excel(XLSX, sheet_name="Recordings")
codes = pd.read_excel(XLSX, sheet_name="Codes")

print(len(ann), "events in", ann["recording"].nunique(), "recordings")
display(ann.head())
display(ann["class"].value_counts().sort_index())

In [ ]:
# Events whose code covers several regions are comma-separated (e.g. "C5,C7,C9")
ann["code_list"] = ann["code"].astype(str).str.split(",")
ann.explode("code_list")["code_list"].value_counts().head(10)

## 2. A continuous recording with its annotations

Signals are stored in volts. Annotations are embedded in the file (`raw.annotations`) and are the same as in the workbook.

In [ ]:
name = rec["fif_file"].dropna().iloc[0]          # or e.g. "SZ1_filtered_eeg.fif"
raw = mne.io.read_raw_fif(FIF_DIR / name, preload=True, verbose="ERROR")

print(raw)
print("Sampling rate:", raw.info["sfreq"], "Hz")
print("Channels:", raw.ch_names)
print("Duration: %.1f s" % raw.times[-1])

annots = raw.annotations
print(len(annots), "annotations")
pd.DataFrame({"onset_s": annots.onset, "duration_s": annots.duration,
              "description": annots.description}).head(10)

In [ ]:
# Plot the first 30 s (or the first annotated event) with annotations overlaid
start = float(annots.onset[0]) if len(annots) else 0.0
raw.plot(start=max(start - 5, 0), duration=30, n_channels=19, scalings=dict(eeg=100e-6), show=True);

### Extract the signal of one annotated event

In [ ]:
def event_data(raw, onset, duration):
    """Return (channels x samples) array in volts and the time vector for one event."""
    sf = raw.info["sfreq"]
    i0 = int(round(onset * sf))
    i1 = int(round((onset + duration) * sf))
    return raw.get_data(start=i0, stop=i1), raw.times[i0:i1]

# first seizure (class D) event of this recording
mask = np.array([d.startswith("D") for d in annots.description])
if mask.any():
    k = int(np.flatnonzero(mask)[0])
    data, t = event_data(raw, annots.onset[k], annots.duration[k])
    print(annots.description[k], data.shape)

## 3. One-second windows

Arrays: `X` (windows x 19 channels x 250 samples, volts), `y` (class A–D), `recording_id`, `start_s` (window start in seconds from the start of the recording) and `win_mode` (`train` or `test`).

In [ ]:
npz = np.load(NPZ, allow_pickle=False)
print({k: npz[k].shape for k in npz.files})
print(npz.files)

In [ ]:
# Loading X needs about 1.4 GB of memory. To avoid that, open it lazily and take a slice:
X = npz["X"]
y = npz["y"].astype(str)
recording_id = npz["recording_id"]
start_s = npz["start_s"]
win_mode = npz["win_mode"].astype(str)

index = pd.read_csv(INDEX, sep="\t")
display(index.head())

summary = (pd.DataFrame({"y": y, "mode": win_mode})
             .value_counts().unstack("mode", fill_value=0))
display(summary)

### Test set and training set

The `test` windows come from a non-overlapping 1 s grid. The `train` windows add augmented windows for class C. Windows from the same recording must not be split between training and test data, so split by `recording_id`.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit   # pip install scikit-learn

test_mask = win_mode == "test"
idx = np.flatnonzero(test_mask)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr, te = next(splitter.split(idx, y[idx], groups=recording_id[idx]))
train_idx, test_idx = idx[tr], idx[te]

assert set(recording_id[train_idx]).isdisjoint(recording_id[test_idx])
print(len(train_idx), "train windows,", len(test_idx), "test windows")

### Plot one window of each class

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(8, 7), sharex=True)
t = np.arange(X.shape[2]) / 250
for ax, c in zip(axes, "ABCD"):
    j = np.flatnonzero(y == c)
    if len(j) == 0:
        continue
    j = j[0]
    for ch in range(5):                     # first 5 channels
        ax.plot(t, X[j, ch] * 1e6 + ch * 150, lw=0.8)
    ax.set_ylabel(f"{c}: {CLASS_NAMES[c]}", rotation=0, ha="right", va="center")
    ax.set_yticks([])
axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

### Link a window back to the continuous recording

In [ ]:
j = np.flatnonzero(y == "D")[0]
rid, t0 = int(recording_id[j]), float(start_s[j])
print(index[index.iloc[:, 0] == rid])
print(f"window starts at {t0:.3f} s in that recording; raw samples {int(round(t0*250))} to {int(round(t0*250))+250}")